# YOLO Comparative Training (v8, v10, v11)

Comparative training of YOLOv8, YOLOv10, and YOLOv11 for GUI element detection.

Each model is trained with identical hyperparameters for a fair comparison.
Results (weights, charts, metrics) are saved in separate folders under `runs/detect/`.

In [ ]:
!pip install ultralytics -q

In [ ]:
import os
from ultralytics import YOLO

## Configuration

In [ ]:
DATA_YAML_PATH = "/content/drive/MyDrive/dataset_split/data.yaml"

IMG_SIZE   = 640
BATCH_SIZE = 16
NUM_EPOCHS = 60
CACHE      = True

MODELS = {
    "yolov8":  "yolov8n.pt",
    "yolov10": "yolov10n.pt",
    "yolov11": "yolo11n.pt",
}

## Training and Validation Functions

In [ ]:
def train_model(model_name, pretrained_weights, data_path):
    """
    Trains a YOLO model with the defined parameters.

    Ultralytics automatically saves:
        - runs/detect/{model_name}/weights/best.pt
        - runs/detect/{model_name}/weights/last.pt
        - runs/detect/{model_name}/results.png
        - runs/detect/{model_name}/results.csv
    """
    print(f"  Starting training: {model_name}")
    print(f"  Base weights: {pretrained_weights}")

    model = YOLO(pretrained_weights)

    results = model.train(
        data=data_path,
        imgsz=IMG_SIZE,
        batch=BATCH_SIZE,
        epochs=NUM_EPOCHS,
        cache=CACHE,
        name=model_name,
        exist_ok=True,
        verbose=True,
    )

    print(f"\n Training of {model_name} complete!")
    print(f"Results at: runs/detect/{model_name}/")

    return results


def validate_model(model_name, data_path):
    """
    Runs validation on the best trained model.
    Prints final metrics (precision, recall, mAP50, mAP50-95).
    """
    weights_path = f"runs/detect/{model_name}/weights/best.pt"

    if not os.path.exists(weights_path):
        print(f" Weights not found: {weights_path}")
        return None

    print(f"\n Validating {model_name} with best.pt...")
    model = YOLO(weights_path)
    metrics = model.val(data=data_path)

    return metrics

## Train All Models

In [ ]:
if not os.path.exists(DATA_YAML_PATH):
    print(f"data.yaml not found: {DATA_YAML_PATH}")
    print("Run data/generate_data_yaml.py first.")
else:
    all_results = {}
    for name, weights in MODELS.items():
        train_model(name, weights, DATA_YAML_PATH)
        metrics = validate_model(name, DATA_YAML_PATH)
        if metrics is not None:
            all_results[name] = metrics

    print("  YOLO MODEL COMPARISON SUMMARY")
    print(f"  {'Model':<12} {'Precision':>10} {'Recall':>10} {'mAP@50':>10} {'mAP@50-95':>12}")

    for name, met in all_results.items():
        p   = met.results_dict.get("metrics/precision(B)", 0)
        r   = met.results_dict.get("metrics/recall(B)", 0)
        m50 = met.results_dict.get("metrics/mAP50(B)", 0)
        m95 = met.results_dict.get("metrics/mAP50-95(B)", 0)
        print(f"  {name:<12} {p:>9.2%} {r:>9.2%} {m50:>9.2%} {m95:>11.2%}")

    print("\n Comparison complete!")
    print(" Charts for each model at: runs/detect/<model_name>/results.png")